### Classifier model using alpha power, FOOOF-derived alpha features, both

Target:
EyesClosed vs EyesOpen

1 = EyesClosed
0 = EyesOpen

#### Imports

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

### Load feature table

In [6]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\maria\Desktop\EEG-special-course")
PROCESSED_ROOT = PROJECT_ROOT / "data_processed"

features_file = PROCESSED_ROOT / "features_pre_ses1.csv"
features = pd.read_csv(features_file)

features.head()

,participant_id,age,condition,alpha_power_8_12,fooof_alpha_cf,fooof_alpha_pw,fooof_alpha_bw,n_epochs_kept,fooof_r2,fooof_error
0,sub-001,60,EyesClosed,-10.268460,10.124218,1.674327,1.942220,91,0.958835,0.087013
1,sub-001,60,EyesOpen,-11.278344,11.232861,0.293344,3.081986,91,0.984296,0.041236
2,sub-002,67,EyesClosed,-10.221186,9.256943,1.711060,2.451334,91,0.964858,0.078536
3,sub-002,67,EyesOpen,-11.416633,8.463996,0.211487,1.000000,91,0.981021,0.053521
4,sub-003,44,EyesClosed,-10.992970,11.359863,1.384598,1.743643,108,0.967652,0.067379


### Define target and helper function

In [7]:
def run_grouped_classifier(df, feature_cols, model_name, n_splits=5):
    df = df.copy()
    df = df.dropna(subset=feature_cols + ["participant_id", "condition"])

    X = df[feature_cols].values
    y = (df["condition"] == "EyesClosed").astype(int).values
    groups = df["participant_id"].values

    cv = GroupKFold(n_splits=n_splits)

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000))
    ])

    scores = cross_validate(
        pipe,
        X,
        y,
        groups=groups,
        cv=cv,
        scoring=["accuracy", "roc_auc"],
        return_train_score=False
    )

    result = {
        "model": model_name,
        "n_rows": len(df),
        "n_participants": df["participant_id"].nunique(),
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "auc_mean": scores["test_roc_auc"].mean(),
        "auc_std": scores["test_roc_auc"].std(),
    }

    return result

### Run models

In [8]:
results = []

results.append(
    run_grouped_classifier(
        features,
        feature_cols=["alpha_power_8_12"],
        model_name="alpha_8_12_only"
    )
)

results.append(
    run_grouped_classifier(
        features,
        feature_cols=["fooof_alpha_cf", "fooof_alpha_pw", "fooof_alpha_bw"],
        model_name="fooof_only"
    )
)

results.append(
    run_grouped_classifier(
        features,
        feature_cols=["alpha_power_8_12", "fooof_alpha_cf", "fooof_alpha_pw", "fooof_alpha_bw"],
        model_name="combined"
    )
)

results_df = pd.DataFrame(results)
results_df

,model,n_rows,n_participants,accuracy_mean,accuracy_std,auc_mean,auc_std
0,alpha_8_12_only,1215,608,0.739047,0.024259,0.788629,0.017074
1,fooof_only,1092,596,0.793025,0.015102,0.867973,0.008838
2,combined,1092,596,0.802191,0.015804,0.892230,0.009345
